# 04 - Gold Layer: Star Schema
Builds the business-ready star schema from Silver.

| Table | Type | Grain |
|---|---|---|
| `dim_customer` | SCD Type 2 (hand-written MERGE) | One row per customer version |
| `dim_product` | SCD Type 1 (MERGE upsert) | One row per product |
| `dim_date` | Calendar | One row per day |
| `fact_orders` | Fact | One row per order line item |

Safe to re-run: dimensions use MERGE, so nothing is duplicated.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_lakehouse.gold.dim_customer (
  customer_sk              BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key',
  customer_id              STRING NOT NULL COMMENT 'Natural key from source',
  customer_unique_id       STRING,
  customer_zip_code_prefix STRING,
  customer_city            STRING,
  customer_state           STRING,
  row_hash                 STRING COMMENT 'Hash of tracked columns, used to detect changes',
  effective_from           TIMESTAMP COMMENT 'When this version became valid',
  effective_to             TIMESTAMP COMMENT 'When this version stopped being valid (9999-12-31 = current)',
  is_current               BOOLEAN
)
COMMENT 'Gold customer dimension - SCD Type 2. Tracks changes to zip, city and state.';

CREATE TABLE IF NOT EXISTS retail_lakehouse.gold.dim_product (
  product_sk                 BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key',
  product_id                 STRING NOT NULL COMMENT 'Natural key from source',
  product_category_name      STRING,
  product_photos_qty         INT,
  product_weight_g           INT,
  product_length_cm          INT,
  product_height_cm          INT,
  product_width_cm           INT,
  product_volume_cm3         INT,
  row_hash                   STRING,
  updated_at                 TIMESTAMP
)
COMMENT 'Gold product dimension - SCD Type 1 (latest values only).';

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_date
COMMENT 'Gold calendar dimension, one row per day 2016-2027.'
AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d                                       AS date,
  year(d)                                 AS year,
  quarter(d)                              AS quarter,
  month(d)                                AS month,
  date_format(d, 'MMMM')                  AS month_name,
  day(d)                                  AS day_of_month,
  dayofweek(d)                            AS day_of_week,
  date_format(d, 'EEEE')                  AS day_name,
  dayofweek(d) IN (1, 7)                  AS is_weekend
FROM (
  SELECT explode(sequence(DATE'2016-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS d
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW products_src AS
SELECT
  product_id,
  product_category_name,
  product_photos_qty,
  product_weight_g,
  product_length_cm,
  product_height_cm,
  product_width_cm,
  product_length_cm * product_height_cm * product_width_cm AS product_volume_cm3,
  sha2(concat_ws('||',
    coalesce(product_category_name, '~'),
    coalesce(CAST(product_photos_qty AS STRING), '~'),
    coalesce(CAST(product_weight_g  AS STRING), '~'),
    coalesce(CAST(product_length_cm AS STRING), '~'),
    coalesce(CAST(product_height_cm AS STRING), '~'),
    coalesce(CAST(product_width_cm  AS STRING), '~')
  ), 256) AS row_hash
FROM retail_lakehouse.silver.products;

MERGE INTO retail_lakehouse.gold.dim_product AS tgt
USING products_src AS src
ON tgt.product_id = src.product_id

-- Product changed -> overwrite (Type 1: no history)
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET
  product_category_name = src.product_category_name,
  product_photos_qty    = src.product_photos_qty,
  product_weight_g      = src.product_weight_g,
  product_length_cm     = src.product_length_cm,
  product_height_cm     = src.product_height_cm,
  product_width_cm      = src.product_width_cm,
  product_volume_cm3    = src.product_volume_cm3,
  row_hash              = src.row_hash,
  updated_at            = current_timestamp()

-- New product -> insert
WHEN NOT MATCHED THEN INSERT (
  product_id, product_category_name, product_photos_qty, product_weight_g,
  product_length_cm, product_height_cm, product_width_cm, product_volume_cm3,
  row_hash, updated_at
) VALUES (
  src.product_id, src.product_category_name, src.product_photos_qty, src.product_weight_g,
  src.product_length_cm, src.product_height_cm, src.product_width_cm, src.product_volume_cm3,
  src.row_hash, current_timestamp()
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_src AS
SELECT
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state,
  sha2(concat_ws('||',
    coalesce(customer_zip_code_prefix, '~'),
    coalesce(customer_city, '~'),
    coalesce(customer_state, '~')
  ), 256) AS row_hash
FROM retail_lakehouse.silver.customers;

MERGE INTO retail_lakehouse.gold.dim_customer AS tgt
USING (
  -- Part A: every source row, matched on its natural key
  SELECT customer_id AS merge_key, * FROM customers_src

  UNION ALL

  -- Part B: CHANGED customers again, with a NULL key so they can't match
  --         -> forces an INSERT of the new version
  SELECT NULL AS merge_key, src.*
  FROM customers_src src
  JOIN retail_lakehouse.gold.dim_customer cur
    ON src.customer_id = cur.customer_id
   AND cur.is_current = true
  WHERE src.row_hash <> cur.row_hash
) AS stg
ON tgt.customer_id = stg.merge_key AND tgt.is_current = true

-- Customer changed -> close the old version
WHEN MATCHED AND tgt.row_hash <> stg.row_hash THEN UPDATE SET
  is_current   = false,
  effective_to = current_timestamp()

-- New customer, or new version of a changed customer -> insert
WHEN NOT MATCHED THEN INSERT (
  customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state,
  row_hash, effective_from, effective_to, is_current
) VALUES (
  stg.customer_id, stg.customer_unique_id, stg.customer_zip_code_prefix, stg.customer_city, stg.customer_state,
  stg.row_hash,
  -- first version is valid "forever back"; later versions start now
  CASE WHEN stg.merge_key IS NULL THEN current_timestamp() ELSE TIMESTAMP'1900-01-01' END,
  TIMESTAMP'9999-12-31',
  true
);

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.fact_orders
CLUSTER BY (purchase_date_key, product_sk)
COMMENT 'Gold fact table. Grain: one row per order line item. Customer key is point-in-time (SCD2). Clustered for date and product filters.'
AS
SELECT
  oi.order_id,
  oi.order_item_id,
  dc.customer_sk,
  dp.product_sk,
  CAST(date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS purchase_date_key,
  oi.seller_id,
  o.order_status,
  o.order_purchase_timestamp,
  o.order_delivered_customer_date,
  o.order_estimated_delivery_date,
  CAST(oi.price AS DECIMAL(10,2))                     AS price,
  CAST(oi.freight_value AS DECIMAL(10,2))             AS freight_value,
  CAST(oi.price + oi.freight_value AS DECIMAL(10,2))  AS item_total,
  datediff(o.order_delivered_customer_date, o.order_purchase_timestamp) AS delivery_days,
  CASE
    WHEN o.order_delivered_customer_date IS NULL THEN NULL
    ELSE o.order_delivered_customer_date > o.order_estimated_delivery_date
  END AS is_late_delivery
FROM retail_lakehouse.silver.order_items oi
JOIN retail_lakehouse.silver.orders o
  ON oi.order_id = o.order_id
-- Point-in-time join: pick the customer version that was valid WHEN the order was placed
LEFT JOIN retail_lakehouse.gold.dim_customer dc
  ON o.customer_id = dc.customer_id
 AND o.order_purchase_timestamp >= dc.effective_from
 AND o.order_purchase_timestamp <  dc.effective_to
LEFT JOIN retail_lakehouse.gold.dim_product dp
  ON oi.product_id = dp.product_id;

In [0]:
%sql
SELECT 'dim_customer' AS table_name, COUNT(*) AS row_count FROM retail_lakehouse.gold.dim_customer
UNION ALL SELECT 'dim_product', COUNT(*) FROM retail_lakehouse.gold.dim_product
UNION ALL SELECT 'dim_date',    COUNT(*) FROM retail_lakehouse.gold.dim_date
UNION ALL SELECT 'fact_orders', COUNT(*) FROM retail_lakehouse.gold.fact_orders;

In [0]:
%sql
-- Cluster dimensions on their lookup keys (safe to run every time)
ALTER TABLE retail_lakehouse.gold.dim_customer CLUSTER BY (customer_id);
ALTER TABLE retail_lakehouse.gold.dim_product  CLUSTER BY (product_id);

-- Compact and cluster all Gold tables
OPTIMIZE retail_lakehouse.gold.fact_orders;
OPTIMIZE retail_lakehouse.gold.dim_customer;
OPTIMIZE retail_lakehouse.gold.dim_product;